In [0]:
# 1) Leer la capa SILVER (datos limpios)
from pyspark.sql import functions as F

SCHEMA = "credito_riesgo_lab"
TABLA_SILVER = f"workspace.{SCHEMA}.clientes_silver"

silver = spark.table(TABLA_SILVER)
print(f"Silver: {silver.count()} filas")

In [0]:
# 2) Enriquecimiento con una dimensión de sector — primer borrador, SIN optimizar
# dim_sector es la "dimensión" (tabla pequeña, 6 filas); silver es la "hecho" (tabla grande)
# En un modelo estrella de banco: hechos = transacciones/movimientos, dimensiones = catálogos pequeños.
dim_sector = spark.sql(f"""
    SELECT sector,
           CASE sector WHEN 'tecnologia' THEN 'bajo'
                       WHEN 'servicios'  THEN 'medio'
                       ELSE 'alto' END AS riesgo_sectorial
    FROM (SELECT DISTINCT sector FROM {TABLA_SILVER})
""")

silver_enriquecido = silver.join(dim_sector, "sector")   # versión ingenua: SIN broadcast

# El plan físico (explain) revela cómo ejecuta Spark este join por dentro:
#   SortMergeJoin + Exchange → ordena y MUEVE ambas tablas por la red (shuffle) = caro
silver_enriquecido.explain()

In [0]:
# 2b) 🚀 MEJORA: broadcast(dim_sector) — el join sin mover la tabla grande
# dim_sector es PEQUEÑA (6 filas): broadcast() la copia íntegra a cada máquina del
# cluster, y el join se hace en LOCAL → silver (la grande) ya no se mueve por la red.
silver_enriquecido = silver.join(F.broadcast(dim_sector), "sector")

# El plan físico cambia: ahora busca "BroadcastHashJoin" (y el Exchange de silver desaparece)
silver_enriquecido.explain()
print("Antes: SortMergeJoin (mueve AMBAS tablas) → Ahora: BroadcastHashJoin (solo copia la pequeña)")

In [0]:
# 3) GOLD (nivel cliente) — score de riesgo con reglas de negocio
# Puntos de riesgo (cuanto más alto, peor):
#   - historial de pagos: moroso +40, atrasos_ocasionales +20
#   - endeudamiento (DTI): hasta +30 · ltv: hasta +20
#   - antigüedad < 12 meses: +10 (cliente nuevo = más incierto)
puntos_riesgo = (
    F.when(F.col("historial_pagos") == "moroso", 40)
     .when(F.col("historial_pagos") == "atrasos_ocasionales", 20)
     .otherwise(0)
    + F.round(F.col("endeudamiento") * 30)
    + F.round(F.col("ltv") * 20)
    + F.when(F.col("antiguedad_meses") < 12, 10).otherwise(0)
)

gold = (
    silver_enriquecido
    .withColumn("score_riesgo", puntos_riesgo.cast("int"))
    .withColumn("nivel_riesgo",
        F.when(F.col("score_riesgo") >= 55, F.lit("alto"))
         .when(F.col("score_riesgo") >= 30, F.lit("medio"))
         .otherwise(F.lit("bajo")))
)

TABLA_GOLD_CLIENTES = f"workspace.{SCHEMA}.clientes_gold"
(
    gold.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_GOLD_CLIENTES)
)

display(gold.groupBy("nivel_riesgo").count().orderBy("nivel_riesgo"))

In [0]:
# 4) GOLD (nivel negocio) — resumen por sector, listo para dashboards
resumen = (
    gold.groupBy("sector")
    .agg(
        F.count("*").alias("clientes"),
        F.round(F.avg("ingresos"), 2).alias("ingreso_medio"),
        F.round(F.avg("score_riesgo"), 1).alias("score_medio"),
        F.round(F.avg(F.when(F.col("nivel_riesgo") == "alto", 1).otherwise(0)) * 100, 1)
            .alias("pct_riesgo_alto"),
    )
    .orderBy(F.desc("pct_riesgo_alto"))
)

TABLA_GOLD_RESUMEN = f"workspace.{SCHEMA}.resumen_riesgo_sector"
(
    resumen.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_GOLD_RESUMEN)
)

display(resumen)

In [0]:
# 5) Verificación final: la cadena completa landing → bronze → silver → gold
for t in ["landing_clientes", "clientes_bronze", "clientes_silver", "clientes_gold", "resumen_riesgo_sector"]:
    n = spark.table(f"workspace.{SCHEMA}.{t}").count()
    print(f"workspace.{SCHEMA}.{t:<24} → {n} filas")

display(spark.table(f"workspace.{SCHEMA}.clientes_gold").limit(5))

# 03 · Capa GOLD — Reglas de negocio

**Qué hace una capa gold en la industria:** aplicar **reglas de negocio** sobre los datos limpios y dejar tablas listas para consumo (dashboards, ML, reporting). Aquí: score y nivel de riesgo por cliente, enriquecimiento con dimensión de sector y resumen por sector.

**🚀 Optimización integrada (como en producción):** primero el join SIN broadcast (versión ingenua), miramos el plan físico, y luego mejoramos con `broadcast()` → sin mover la tabla grande por la red.

> Lee `clientes_silver` y escribe `clientes_gold` + `resumen_riesgo_sector`.